# Data Preprocessing of DEID_RX_ORDER.csv using SQL

# 0. Environment Setup & Library Imports

We initialize the necessary libraries for high-performance data processing. DuckDB is used for out-of-core SQL operations to handle the massive dataset without overloading RAM, while PyArrow enables efficient Parquet file storage. Regular Expressions (re) are loaded for advanced text normalization.

In [ ]:
"""
Created on Tuesday Nov 11 2025

@author: Laura Rueda
"""

import pandas as pd
import os
import duckdb
import re
import gc

# 1. DuckDB Cleaning: ETL Pipeline with DuckDB (Cleaning & Sorting)

The raw dataset contains over 20 million rows, causing memory overflows with standard Pandas loading. We utilize DuckDB to process the file in a streaming fashion directly from the disk. This step cleans missing values (imputation), fixes data types, and performs the critical step of sorting by PATIENT_ID and Shifted_date, ensuring the temporal sequence required for the LSTM Neural Network is preserved.

duckdb is a python library that doesn't need server installation and works inside the notebook but with the power of sql: "In-process OLAP database".


In [3]:
input_file_sql = 'data/deid_rx_order.csv'        # Input file path
output_file_sql = 'data/deid_rx_order_final_sql.csv'      # Final clean file

In [5]:
# --- ALTERNATIVE EFFICIENT METHOD USING DUCKDB ---
print("Using SQL (DuckDB) to clean, order and save the data...")

# Connect to the temporary database
con = duckdb.connect()
print("   DuckDB connected.")

# This query does ALL the work of Pandas but without using RAM:
# 1. Reads the temporary file
# 2. Fills nulls (COALESCE)
# 3. Orders by PATIENT_ID and Shifted_date
# 4. Saves the final result directly to disk
query = f"""
COPY (
    SELECT 
        AIM_GROUP, PATIENT_ID, ENCOUNTER_ID, RX_CODE, RX_NAME,
        COALESCE(ROUTE, 'Unknown') AS ROUTE,
        COALESCE(DOSE, 'Unknown') AS DOSE,
        COALESCE(FREQUENCY, 'Unknown') AS FREQUENCY,
        CAST(Shifted_date AS TIMESTAMP) AS Shifted_date
    FROM '{input_file_sql}'
    ORDER BY PATIENT_ID, Shifted_date
) TO '{output_file_sql}' (HEADER, DELIMITER ',')
"""

con.execute(query)
print(f"¡Ready! Clean file saved at: {output_file_sql}")

# --- IMPORTANT (Top 50) ---
# Since DuckDB saves directly to disk, the variable 'df' does not exist in memory.
# We load only the DOSE column so that your next Top 50 cell works:
print("Loading DOSE column for analysis...")
df = pd.read_csv(output_file_sql, usecols=['DOSE'])

Using SQL (DuckDB) to clean, order and save the data...
   DuckDB connected.
¡Ready! Clean file saved at: data/deid_rx_order_final_sql.csv
Loading DOSE column for analysis...


# 2. One-Hot-Encoding of DOSE column

#### 2.1 Dose Normalization (Regex & Math)

The DOSE column contains unstructured clinical text with high variability (e.g., "1g", "1000 mg", "1,000mg"). We apply a custom normalization function using Regular Expressions to standardize units and formats. This includes mathematical conversion (e.g., converting grams to milligrams) to unify equivalent dosages, significantly increasing the data coverage before encoding.

1st way: 79.71%

In [12]:
def normalize_dose_smart(value):
    if pd.isna(value) or value == 'Unknown':
        return 'Unknown'
    
    # Lowercase and strip spaces
    value = str(value).lower().strip()
    
    # 2. Remove thousand separators ("1,000 mg" -> "1000 mg")
    value = value.replace(',', '')
    
    # 3. REGULAR EXPRESSIONS: It looks for a number (integer or decimal) followed by anything
    match = re.match(r"(\d*\.?\d+)\s*([a-z]+.*)", value)
    
    if match:
        # match.group(1) is the number, match.group(2) is the unit
        number = match.group(1)
        unit = match.group(2).strip()
        
        # 4. UNIT STANDARDIZATION 
        unit_mapping = {
            'gm': 'g', 'gms': 'g', 'gram': 'g', 'grams': 'g',
            'mg': 'mg', 'mgs': 'mg', 'milligrams': 'mg',
            'mcg': 'mcg', 'micrograms': 'mcg',
            'ml': 'ml', 'mls': 'ml',
            'unit': 'units', 'u': 'units',
            'tablet': 'tab', 'tablets': 'tab', 'pill': 'tab'
        }
        
        clean_unit = unit_mapping.get(unit, unit)
        
        # 5. Rebuild clean string: "NUMBER + SPACE + UNIT"
        return f"{number} {clean_unit}"
        
    return value # Returns the original if it doesn't match the pattern (e.g., "one time")


print("Applying smart cleaning (Regex)...")
df['DOSE_CLEAN'] = df['DOSE'].astype(str).apply(normalize_dose_smart)

# Compare some examples before and after cleaning
print("\nExamples of improvements:")
print(df[['DOSE', 'DOSE_CLEAN']].sample(10))

Applying smart cleaning (Regex)...

Examples of improvements:
             DOSE DOSE_CLEAN
18883818   750 mg     750 mg
19627562  Unknown    Unknown
11378718   400 mg     400 mg
253760     500 mg     500 mg
283406    Unknown    Unknown
19754064     1 MG       1 mg
2332466    100 mg     100 mg
2830930    100 mg     100 mg
5168381    200 mg     200 mg
4299237     50 mg      50 mg


2nd way: 83.00%

In [14]:
def normalize_dose_math(value):
    if pd.isna(value) or value == 'Unknown':
        return 'Unknown'
    
    # 1. Basic cleaning
    value = str(value).lower().strip()
    value = value.replace(',', '') # Remove thousand separators
    
    # 2. Regex to separate Number and Unit
    match = re.match(r"(\d*\.?\d+)\s*([a-z]+.*)", value)
    
    if match:
        try:
            number = float(match.group(1)) # Convert to real number for operations
            unit = match.group(2).strip()
            
            # 3. Standardize unit names (Singular/Plural/Synonyms)
            unit_mapping = {
                'gm': 'g', 'gms': 'g', 'gram': 'g', 'grams': 'g',
                'mg': 'mg', 'mgs': 'mg', 'milligrams': 'mg',
                'mcg': 'mcg', 'micrograms': 'mcg',
                'ml': 'ml', 'mls': 'ml',
                'l': 'l', 'liter': 'l', 'liters': 'l',
                'unit': 'units', 'u': 'units',
                'tablet': 'tab', 'tablets': 'tab', 'pill': 'tab', 'cap': 'cap', 'capsule': 'cap'
            }
            unit = unit_mapping.get(unit, unit)
            
            # 4. MATHEMATICAL CONVERSION (The trick to gain coverage)
            # Convert everything to the most common unit: mg and ml
            
            if unit == 'g':        # Grams to Milligrams
                number *= 1000
                unit = 'mg'
            elif unit == 'mcg':    # Micrograms to Milligrams
                number /= 1000
                unit = 'mg'
            elif unit == 'l':      # Liters to Milliliters
                number *= 1000
                unit = 'ml'
            elif unit == 'kg':     # Kilograms to Grams (rare in doses, but possible)
                number *= 1000
                unit = 'g'
                
            # 5. Intelligent Formatting (Remove unnecessary decimals)
            # "1000.0 mg" becomes "1000 mg"
            if number.is_integer():
                clean_number = str(int(number))
            else:
                # Optional: Round crazy decimals (e.g., 0.33333 -> 0.33)
                clean_number = str(round(number, 3))
                # Remove trailing zeros '0.50' -> '0.5'
                if '.' in clean_number:
                     clean_number = clean_number.rstrip('0').rstrip('.')

            return f"{clean_number} {unit}"

        except ValueError:
            return value # If mathematical conversion fails, return original

    return value


# --- TEST THE IMPROVEMENT ---
print("Applying advanced mathematical normalization...")
df['DOSE_CLEAN'] = df['DOSE'].astype(str).apply(normalize_dose_math)

# Recalculate the Top 50 with the new data
conteo = df['DOSE_CLEAN'].value_counts()
top_50_coverage = conteo.head(50).sum() / len(df) * 100

print(f"\nNew Coverage of the TOP 50: {top_50_coverage:.2f}%")
print("\nExamples of achieved merges:")
# See cases where the original changed
changes = df[df['DOSE'] != df['DOSE_CLEAN']][['DOSE', 'DOSE_CLEAN']].sample(10)
print(changes)

Applying advanced mathematical normalization...

New Coverage of the TOP 50: 83.96%

Examples of achieved merges:

New Coverage of the TOP 50: 83.96%

Examples of achieved merges:
                 DOSE  DOSE_CLEAN
13790017      300 mcg      0.3 mg
5999225          1 Gm     1000 mg
11927631  5,000 units  5000 units
8468459      2,625 mg     2625 mg
12455624  5,000 Units  5000 units
17682210         2 Gm     2000 mg
14373929        50 MG       50 mg
8123034      4 MG/2ML    4 mg/2ml
9595107   5,000 units  5000 units
18874470  5,000 Units  5000 units
                 DOSE  DOSE_CLEAN
13790017      300 mcg      0.3 mg
5999225          1 Gm     1000 mg
11927631  5,000 units  5000 units
8468459      2,625 mg     2625 mg
12455624  5,000 Units  5000 units
17682210         2 Gm     2000 mg
14373929        50 MG       50 mg
8123034      4 MG/2ML    4 mg/2ml
9595107   5,000 units  5000 units
18874470  5,000 Units  5000 units


3rd way: 83.96%


In [15]:
def normalize_dose_ultimate(value):
    """
    Ultimate normalization: Handles mEq, sprays, puffs, and standard math.
    """
    if pd.isna(value) or value == 'Unknown':
        return 'Unknown'
    
    # 1. Basic Cleaning
    value = str(value).lower().strip()
    value = value.replace(',', '')
    
    # 2. Regex: Allow for number followed by potentially weird chars (like / or -)
    # This captures "number" + "space" + "unit"
    match = re.match(r"(\d*\.?\d+)\s*([a-z]+.*)", value)
    
    if match:
        try:
            number = float(match.group(1))
            unit = match.group(2).strip()
            
            # 3. MASTER MAPPING (The secret sauce)
            unit_mapping = {
                # Weight
                'gm': 'g', 'gms': 'g', 'gram': 'g', 'grams': 'g',
                'mg': 'mg', 'mgs': 'mg', 'milligrams': 'mg',
                'mcg': 'mcg', 'micrograms': 'mcg',
                'kg': 'kg', 'kilograms': 'kg',
                
                # Volume
                'ml': 'ml', 'mls': 'ml', 'milliliters': 'ml',
                'l': 'l', 'liter': 'l', 'liters': 'l',
                
                # Units / Potency
                'unit': 'units', 'u': 'units', 'iu': 'units', 'intl units': 'units',
                'meq': 'meq', 'mEq': 'meq', 'milliequivalents': 'meq', # KEEP THIS!
                
                # Forms
                'tablet': 'tab', 'tablets': 'tab', 'tab': 'tab', 'pill': 'tab', 'pills': 'tab',
                'cap': 'cap', 'capsule': 'cap', 'caps': 'cap',
                'patch': 'patch', 'patches': 'patch',
                'spray': 'spray', 'sprays': 'spray', 'puff': 'spray', 'puffs': 'spray',
                'app': 'app', 'appl': 'app', 'application': 'app',
                'vial': 'vial', 'amp': 'vial', 'ampul': 'vial'
            }
            
            # Normalize the unit name
            # We split by space to handle cases like "mg tablet" -> just "mg"
            first_word = unit.split()[0]
            unit = unit_mapping.get(first_word, first_word)
            
            # 4. MATH CONVERSIONS
            if unit == 'g': number *= 1000; unit = 'mg'
            elif unit == 'mcg': number /= 1000; unit = 'mg'
            elif unit == 'l': number *= 1000; unit = 'ml'
            elif unit == 'kg': number *= 1000; unit = 'g'
                
            # 5. Formatting
            if number.is_integer():
                clean_number = str(int(number))
            else:
                clean_number = str(round(number, 3))
                if '.' in clean_number:
                     clean_number = clean_number.rstrip('0').rstrip('.')

            return f"{clean_number} {unit}"

        except ValueError:
            return value

    return value

# --- EJECUTAR ---
print("Applying ULTIMATE normalization...")
df['DOSE_CLEAN'] = df['DOSE'].astype(str).apply(normalize_dose_ultimate)

# Check coverage
counts = df['DOSE_CLEAN'].value_counts()
top_50_coverage = counts.head(50).sum() / len(df) * 100
print(f"\nNew Coverage: {top_50_coverage:.2f}%")

Applying ULTIMATE normalization...

New Coverage: 84.20%

New Coverage: 84.20%


#### 2.2 Dose Encoding: High-Cardinality Strategy (Top-50 One-Hot)

To manage the high dimensionality of dosage data, we implement a "Top-N + Other" strategy. We isolate the 50 most frequent dosages and bin the remaining tail into an "Other" category. The resulting categorical feature is transformed into a One-Hot Encoded matrix (int8) and saved as a Parquet file for optimized storage and fast retrieval.

In [16]:
# 1. Calculate how many times each dosis appears
# value_counts() counts and orders from most to least frequent
conteo_dosis = df['DOSE_CLEAN'].value_counts()
# Save the complete list to a text file
conteo_dosis.to_string('complete_doses_list.txt')

print("Complete list of different doses saved in 'complete_doses_list.txt'.")

# 2. Show only the TOP 50
print("--- TOP 50 MOST FREQUENT DOSES ---")
display(conteo_dosis.head(50))

# 3. Calculate the coverage of these TOP 50 doses
total_rows = len(df)
rows_top_50 = conteo_dosis.head(50).sum()
print(f"\nThese 50 doses cover {rows_top_50} rows.")
print(f"They represent {(rows_top_50 / total_rows) * 100:.2f}% of all your data.")

Complete list of different doses saved in 'complete_doses_list.txt'.
--- TOP 50 MOST FREQUENT DOSES ---


DOSE_CLEAN
Unknown       4318435
100 mg        1048542
10 mg          817334
20 mg          671198
5000 units     631998
500 mg         618805
4 mg           499481
40 mg          458704
1 gtt          457050
5 mg           449280
50 mg          443673
1000 mg        410077
25 mg          407808
30 mg          406351
1 mg           344413
600 mg         323817
2 mg           293678
300 mg         282068
1 app          280456
325 mg         261670
2000 mg        257697
200 mg         227461
150 mg         181723
2.5 mg         178799
400 mg         178696
15 mg          159141
80 mg          158569
650 mg         147377
2 spray        144323
0.2 mg         144236
800 mg         143887
60 mg          142543
0.025 mg       133476
81 mg          130496
1 patch        123922
250 mg         114280
3375 mg        101377
0.1 mg         100248
0.5 mg          98155
0.05 mg         97476
0.4 mg          95907
17000 mg        95019
3 ml            94436
12.5 mg         91251
8 mg            82774


These 50 doses cover 17208284 rows.
They represent 84.20% of all your data.


In [17]:
# --- STEP 1: DEFINE THE TOP 50 LIST ---
# We extract the names of the 50 most frequent doses from your cleaned column
top_50_doses = df['DOSE_CLEAN'].value_counts().nlargest(50).index

# --- STEP 2: REDUCE CARDINALITY (The "Other" Bucket) ---
# Logic: If the dose is in the Top 50, keep it. If not, replace it with 'Other'.
# We use numpy/pandas vectorization (.where) which is much faster than a loop.
df['DOSE_REDUCED'] = df['DOSE_CLEAN'].where(df['DOSE_CLEAN'].isin(top_50_doses), 'Other')

# Check that it worked (you should only see 51 unique values now)
print(f"Unique categories after reduction: {df['DOSE_REDUCED'].nunique()} (Target: 51)")

# --- STEP 3: GENERATE ONE-HOT ENCODING ---
# This creates the binary columns (0s and 1s).
# CRITICAL: dtype='int8' forces Python to use 1 byte per number instead of 8. 
# This reduces memory usage by 87% (Vital for 20 million rows).
dose_dummies = pd.get_dummies(df['DOSE_REDUCED'], prefix='DOSE', dtype='int8')

# --- STEP 4: PREVIEW ---
print("\nOne-Hot Matrix Shape:", dose_dummies.shape)
print(dose_dummies.head())

# NOTE: The 'dose_dummies' variable now holds your features for the Neural Network.
# Since you have 20M rows, avoid doing "pd.concat([df, dose_dummies])" unless you have >32GB RAM.
# It's safer to save 'dose_dummies' or use it directly in batches.

Unique categories after reduction: 51 (Target: 51)

One-Hot Matrix Shape: (20436701, 51)

One-Hot Matrix Shape: (20436701, 51)
   DOSE_0.025 mg  DOSE_0.05 mg  DOSE_0.1 mg  DOSE_0.2 mg  DOSE_0.4 mg  \
0              0             0            0            0            0   
1              0             0            0            0            0   
2              0             0            0            0            0   
3              0             0            0            0            0   
4              0             1            0            0            0   

   DOSE_0.5 mg  DOSE_1 app  DOSE_1 gtt  DOSE_1 mg  DOSE_1 patch  ...  \
0            0           0           0          0             0  ...   
1            0           0           0          0             0  ...   
2            0           0           0          1             0  ...   
3            0           0           0          0             0  ...   
4            0           0           0          0             0  ...   

 

Dependencies needed for using Parquet:
- pip install pyarrow
- pip install fastparquet

In [ ]:
# --- STEP 5: SAVE THE ONE-HOT ENCODING OF DOSES  ---
print("Saving dose matrix to disk (Parquet format)...")
os.makedirs('data/onehotencoding', exist_ok=True)
dose_dummies.to_parquet('data/onehotencoding/dose_one_hot.parquet', engine='pyarrow')

print("Saved!")

# Free memory now that it's saved
del dose_dummies
gc.collect()
print("Memory freed.")

# 3. Medication Encoding: Label Encoding for Embeddings (RX_CODE)

The dataset contains over 3,000 unique medication identifiers (RX_CODE). Applying standard One-Hot Encoding here would be computationally prohibitive, resulting in a massive sparse matrix (over 60GB). Instead, we employ Label Encoding to map each unique medication code to a sequential integer ID. These IDs will serve as inputs for a learnable Embedding Layer in the LSTM network, allowing the model to efficiently capture semantic relationships between different drugs in a low-dimensional vector space.

In [ ]:

# --- STEP 1: LOAD ONLY RX_CODE ---
print("Loading RX_CODE column...")
df_meds = pd.read_csv('data/deid_rx_order.csv', usecols=['RX_CODE'])

# --- STEP 2: GENERATE NUMERIC IDs (Label Encoding) ---
print(f"Processing {df_meds['RX_CODE'].nunique()} unique medications...")

# Convert to 'category' type to automatically assign an integer ID to each unique code
df_meds['RX_CODE_ID'] = df_meds['RX_CODE'].astype('category').cat.codes

# --- STEP 3: SAVE THE MAPPING (Important!) ---
# You need a dictionary to know that ID 105 corresponds to "Ibuprofen"
print("Creating mapping dictionary...")
mapping = df_meds[['RX_CODE', 'RX_CODE_ID']].drop_duplicates().sort_values('RX_CODE_ID')
mapping.to_csv('data/rx_code_mapping.csv', index=False)
print("Mapping saved to 'data/rx_code_mapping.csv'.")

# --- STEP 4: SAVE THE PROCESSED IDs ---
# We save only the ID column. This is the input for the Neural Network's Embedding Layer.
print("Saving medication IDs to disk...")
# We save as DataFrame to keep the column name
df_meds[['RX_CODE_ID']].to_parquet('data/rx_code_ids.parquet', engine='pyarrow')
print("Encoded IDs saved to 'data/rx_code_ids.parquet'.")

# --- CLEANUP ---
del df_meds, mapping
gc.collect()
print("Memory released. Medication processing complete.")

Loading RX_CODE column...
Processing 13290 unique medications...
Processing 13290 unique medications...
Creating mapping dictionary...
Creating mapping dictionary...
✅ Mapping saved to 'data/rx_code_mapping.csv'.
Saving medication IDs to disk...
✅ Mapping saved to 'data/rx_code_mapping.csv'.
Saving medication IDs to disk...
✅ Encoded IDs saved to 'data/rx_code_ids.parquet'.
Memory released. Medication processing complete.
✅ Encoded IDs saved to 'data/rx_code_ids.parquet'.
Memory released. Medication processing complete.


# 4. Route Encoding (Top-20 Strategy)

The ROUTE feature (administration path: Oral, IV...) has relatively low cardinality compared to dosages. We apply a Top-20 reduction strategy, which captures the vast majority of administration methods (approx 90% coverage). The remaining rare routes are grouped into an "Other" category. Finally, the feature is vectorized using One-Hot Encoding and saved as an independent Parquet module (route_one_hot.parquet).

In [8]:
# --- STEP 1: LOAD ROUTE COLUMN ---
print("Loading ROUTE column...")
# We only load 'ROUTE' to be efficient
df_route = pd.read_csv('data/deid_rx_order.csv', usecols=['ROUTE'])

# --- STEP 2: ANALYZE AND REDUCE ---
# Let's look at the Top 20 to see what we have
top_20_routes = df_route['ROUTE'].value_counts().nlargest(20).index
print(f"Top 5 Routes identified: {list(top_20_routes[:5])}")

# Apply "Top 20 + Other" Strategy
# If it's not in the top 20, we label it as 'Other'
df_route['ROUTE_REDUCED'] = df_route['ROUTE'].where(df_route['ROUTE'].isin(top_20_routes), 'Other')

# --- STEP 3: ONE-HOT ENCODING ---
print("Generating One-Hot matrix...")
route_dummies = pd.get_dummies(df_route['ROUTE_REDUCED'], prefix='ROUTE', dtype='int8')

print(f"Matrix Shape: {route_dummies.shape}")

# --- STEP 4: SAVE TO DISK ---
print("Saving to 'data/onehotencoding/route_one_hot.parquet'...")
os.makedirs('data/onehotencoding', exist_ok=True)
route_dummies.to_parquet('data/onehotencoding/route_one_hot.parquet', engine='pyarrow')


Loading ROUTE column...
Top 5 Routes identified: ['PO', 'IV', 'IV PUSH', 'SC', 'IVPB']
Generating One-Hot matrix...
Matrix Shape: (20436701, 21)
Saving to 'data/onehotencoding/route_one_hot.parquet'...


In [27]:
# 3. Calculate the coverage of these TOP 20 routes
total_rows = len(df_route)
rows_top_20 = df_route['ROUTE'].value_counts().nlargest(20).sum()
print(f"\nThese 20 routes cover {rows_top_20} rows.")
print(f"They represent {(rows_top_20 / total_rows) * 100:.2f}% of all your data.")



These 20 routes cover 18298107 rows.
They represent 89.54% of all your data.


In [ ]:

# --- CLEANUP ---
del df_route, route_dummies
gc.collect()
print("✅ Done! ROUTE processing complete and memory released.")

# 5. Frequency Encoding (Top-20 Strategy)

The FREQUENCY column (e.g., Daily, Q4H, BID) often contains variability due to inconsistent data entry (e.g., casing issues). After a basic text normalization step (uppercasing), we implement a Top-20 reduction strategy to capture the most common dosing schedules. This covers the most significant patterns for the model while keeping the feature space manageable. The result is saved as a sparse One-Hot matrix (freq_one_hot.parquet).

In [4]:
# --- STEP 1: LOAD FREQUENCY COLUMN ---
print("Loading FREQUENCY column...")
df_freq = pd.read_csv('data/deid_rx_order.csv', usecols=['FREQUENCY'])

# --- STEP 2: BASIC CLEANING ---
# Frequencies often have casing issues. We convert everything to uppercase to merge them.
df_freq['FREQUENCY'] = df_freq['FREQUENCY'].astype(str).str.upper().str.strip()

# --- STEP 3: ANALYZE AND REDUCE (Top 20) ---
# We check the top 20 most common frequencies
top_20_freq = df_freq['FREQUENCY'].value_counts().nlargest(20).index
print(f"Top 5 Frequencies: {list(top_20_freq[:5])}")

# Apply "Top 20 + Other" Strategy
df_freq['FREQ_REDUCED'] = df_freq['FREQUENCY'].where(df_freq['FREQUENCY'].isin(top_20_freq), 'OTHER')

# Check coverage percentage
coverage = df_freq['FREQ_REDUCED'].value_counts(normalize=True).head(20).sum() * 100
print(f"Top 20 Frequencies cover {coverage:.2f}% of the data.")

Loading FREQUENCY column...
Top 5 Frequencies: ['NAN', 'DAILY', 'ONE TIME ONLY', 'ONE TIME-SPECIFY TIME', 'BID']
Top 20 Frequencies cover 99.51% of the data.


In [7]:
# --- STEP 4: ONE-HOT ENCODING ---
print("Generating One-Hot matrix...")
freq_dummies = pd.get_dummies(df_freq['FREQ_REDUCED'], prefix='FREQ', dtype='int8')

print(f"Matrix Shape: {freq_dummies.shape}")

# --- STEP 5: SAVE TO DISK ---
print("Saving to 'data/freq_one_hot.parquet'...")
os.makedirs('data/onehotencoding', exist_ok=True)
freq_dummies.to_parquet('data/onehotencoding/freq_one_hot.parquet', engine='pyarrow')

Generating One-Hot matrix...
Matrix Shape: (20436701, 21)
Saving to 'data/freq_one_hot.parquet'...


In [ ]:
# --- CLEANUP ---
del df_freq, freq_dummies
gc.collect()
print("FREQUENCY processing complete.")

# Check info is complete in parquets

In [1]:
import pandas as pd

# Cargar el archivo completo en un DataFrame
df = pd.read_parquet('data/onehotencoding/dose_one_hot.parquet')

# Ver las primeras filas
print(df.head())

   DOSE_0.025 mg  DOSE_0.05 mg  DOSE_0.1 mg  DOSE_0.2 mg  DOSE_0.4 mg  \
0              0             0            0            0            0   
1              0             0            0            0            0   
2              0             0            0            0            0   
3              0             0            0            0            0   
4              0             1            0            0            0   

   DOSE_0.5 mg  DOSE_1 app  DOSE_1 gtt  DOSE_1 mg  DOSE_1 patch  ...  \
0            0           0           0          0             0  ...   
1            0           0           0          0             0  ...   
2            0           0           0          1             0  ...   
3            0           0           0          0             0  ...   
4            0           0           0          0             0  ...   

   DOSE_600 mg  DOSE_650 mg  DOSE_75 mg  DOSE_750 mg  DOSE_8 mg  DOSE_80 mg  \
0            0            0           0          

In [2]:
# Cambia 'DOSE_10 mg' por el nombre exacto de una de tus columnas
columna_ejemplo = 'DOSE_10 mg' 

# Muéstrame solo las filas donde esa columna vale 1
ejemplo = df[df[columna_ejemplo] == 1].head(10)

print(f"--- Pacientes que toman {columna_ejemplo} ---")
print(ejemplo)

--- Pacientes que toman DOSE_10 mg ---
     DOSE_0.025 mg  DOSE_0.05 mg  DOSE_0.1 mg  DOSE_0.2 mg  DOSE_0.4 mg  \
42               0             0            0            0            0   
45               0             0            0            0            0   
47               0             0            0            0            0   
70               0             0            0            0            0   
100              0             0            0            0            0   
101              0             0            0            0            0   
188              0             0            0            0            0   
206              0             0            0            0            0   
275              0             0            0            0            0   
277              0             0            0            0            0   

     DOSE_0.5 mg  DOSE_1 app  DOSE_1 gtt  DOSE_1 mg  DOSE_1 patch  ...  \
42             0           0           0          0          

In [3]:
# Guardar las primeras 1000 filas en un archivo para verlas en Excel
df.head(1000).to_csv('muestra_one_hot.csv', index=False)
print("Guardado 'muestra_one_hot.csv'. Ábrelo en Excel para verlo bien.")

Guardado 'muestra_one_hot.csv'. Ábrelo en Excel para verlo bien.


In [4]:
# Cargar solo el número de filas para comprobar
len_original = len(pd.read_csv('data/clean_csv/deid_rx_order_final_sql.csv', usecols=['PATIENT_ID']))
len_dosis = len(pd.read_parquet('data/onehotencoding/dose_one_hot.parquet'))

print(f"Filas originales: {len_original}")
print(f"Filas de dosis:   {len_dosis}")

if len_original == len_dosis:
    print("Coinciden exactamente. Puedes unirlas sin miedo.")
else:
    print("Algo ha pasado, los tamaños son distintos.")

Filas originales: 20436701
Filas de dosis:   20436701
✅ ¡PERFECTO! Coinciden exactamente. Puedes unirlas sin miedo.


In [6]:
import pandas as pd

# Leer solo la columna RX_CODE
df_meds = pd.read_csv('data/clean_csv/deid_rx_order_final_sql.csv', usecols=['RX_CODE'])

num_unicos = df_meds['RX_CODE'].nunique()
print(f"Hay {num_unicos} códigos diferentes de medicamentos.")

Hay 13290 códigos diferentes de medicamentos.


In [7]:
import pandas as pd

# Cargar el archivo completo en un DataFrame
df = pd.read_parquet('data/embeddings/rx_code_ids.parquet')

# Ver las primeras filas
print(df.head())
df.head(1000).to_csv('rx_code_ids_sample.csv', index=False)
print("Guardado 'rx_code_ids_sample.csv'. Ábrelo en Excel para verlo bien.")

   RX_CODE_ID
0        3908
1        3908
2        4253
3        9928
4        9928
Guardado 'rx_code_ids_sample.csv'. Ábrelo en Excel para verlo bien.
